In [ ]:
# Plot Minor Allele Frequency (MAF) and Genotype Missingness 

In [ ]:


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration
maf_file = "analysis_results/variants_analysis/files/maf_data.tsv"
output_dir = "analysis_results/variants_analysis/plots"

# Load and Preprocess MAF Data
maf_cols = ["Chromosome", "Position", "Ref", "Alt", "AC", "AN"]
maf_data = pd.read_csv(maf_file, sep="\t", names=maf_cols)

# Convert to numeric for safety
maf_data["AC"] = pd.to_numeric(maf_data["AC"], errors="coerce")
maf_data["AN"] = pd.to_numeric(maf_data["AN"], errors="coerce")

# Filter: AN >= 5
maf_filtered = maf_data[maf_data["AN"] >= 5].copy()

# Compute MAF as the minor allele frequency
maf_filtered["MAF"] = maf_filtered.apply(lambda row: min(row["AC"], row["AN"] - row["AC"]) / row["AN"], axis=1)

# Filter: MAF > 0.05
maf_filtered = maf_filtered[maf_filtered["MAF"] > 0.05]

# Output stats
print(f"Total sites before filtering: {len(maf_data)}")
print(f"After AN >= 5 filter: {len(maf_data[maf_data['AN'] >= 5])}")
print(f"After MAF > 0.05 filter: {len(maf_filtered)}")

# Optional save
maf_filtered.to_csv("filtered_maf_data.tsv", sep="\t", index=False)


# Plot MAF Histogram
plt.figure(figsize=(8, 5))
sns.histplot(maf_filtered["MAF"], bins=30, kde=True, color="blue")
plt.xlabel("Minor Allele Frequency (MAF)")
plt.ylabel("Count")
plt.title("MAF Distribution (Filtered: AN>=5, MAF>0.05)")
plt.savefig(output_dir/"MAF_distribution_filtered.png", dpi=300, bbox_inches='tight')
plt.show()

# Boxplot of MAF by Chromosome
plt.figure(figsize=(10, 6))
sns.boxplot(data=maf_filtered, x="Chromosome", y="MAF")
plt.xlabel("Chromosome")
plt.ylabel("Minor Allele Frequency (MAF)")
plt.title("MAF Distribution Across Chromosomes (Filtered)")
plt.xticks(rotation=90)
plt.savefig(output_dir/"MAF_distribution_filtered_across_chr.png", dpi=300, bbox_inches='tight')
plt.show()

# Missingness Heatmap from VCF
vcf_path = "/Users/halimaabdulsalam/Plasmodium-WGS-ARISE/PM/Fws/high_quality_annotated_variants.vcf"
vcf_data = pd.read_csv(
    vcf_path, comment='#', sep="\s+",
    usecols=[0,1,3,4] + list(range(9, 19)),
    names=["CHROM", "POS", "REF", "ALT"] + [f"Sample_{i+1}" for i in range(10)]
)

def extract_gt(geno):
    if geno == "." or geno == "./.":
        return np.nan
    return geno.split(":")[0]

vcf_gt = vcf_data.iloc[:, 4:].map(extract_gt)
vcf_gt = vcf_gt.replace([".", "./."], np.nan)

missing_matrix = vcf_gt.notnull()

# Plot
plt.figure(figsize=(14, 8))
sns.heatmap(missing_matrix.T, cmap="Blues", cbar=False, yticklabels=True)
plt.ylabel("Sample ID")
plt.title("Genotype Missingness Heatmap (Plasmodium malariae)")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.savefig(output_dir/"heatmap_missing.png", dpi=300, bbox_inches='tight')
plt.show()
